In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from functools import reduce

In [ ]:
df1= pd.read_excel("id_uni_prot_quan_batch1.xlsx", skiprows = 1)
df1=df1.set_index("Protein Accession #")
df1 = df1.loc[:,df1.columns.str.contains("sig")]

In [ ]:
df2= pd.read_excel("id_uni_prot_quan_batch2.xlsx", skiprows = 1)
df2=df2.set_index("Protein Accession #")
df2 = df2.loc[:,df2.columns.str.contains("sig")]

In [ ]:
df3= pd.read_excel("id_uni_prot_quan_batch3.xlsx", skiprows = 1)
df3=df3.set_index("Protein Accession #")
df3 = df3.loc[:,df3.columns.str.contains("sig")]

In [ ]:
df4= pd.read_excel("id_uni_prot_quan_batch4.xlsx", skiprows = 1)
df4=df4.set_index("Protein Accession #")
df4 = df4.loc[:,df4.columns.str.contains("sig")]

In [ ]:
df5= pd.read_excel("id_uni_prot_quan_batch5.xlsx", skiprows = 1)
df5=df5.set_index("Protein Accession #")
df5 = df5.loc[:,df5.columns.str.contains("sig")]

In [ ]:
df6= pd.read_excel("id_uni_prot_quan_batch6.xlsx", skiprows = 1)
df6=df6.set_index("Protein Accession #")
df6 = df6.loc[:,df6.columns.str.contains("sig")]

In [ ]:
dfs = [df1, df2, df3, df4, df5, df6]

dfs = [
    df.add_prefix(f"batch{i}_")
    for i, df in enumerate(dfs, start=1)
]

df_merged = reduce(
    lambda left, right: left.merge(
        right,
        left_index=True,
        right_index=True,
        how="outer"
    ),
    dfs
)

In [ ]:
df =df_merged

In [ ]:
##info

info = df.loc[:,df.columns.str.contains("sig")].columns.str.split("_").str.get(0)
uniq_batches = info.unique()
n = int(len(info) / len(uniq_batches))


print(df.shape[0], 'total proteins are identified in batchQ.')
print("TMT:",n ,"was used.")
print(len(uniq_batches), 'unique batches are present.')


In [ ]:
# Count non-missing values for each protein across the 6 batches
protein_batch_count = df_merged.notna().sum(axis=1)

# Count how many proteins were identified in exactly 1, 2, ..., 6 batches
summary = (
    protein_batch_count
    .value_counts()
    .sort_index(ascending=False)
)

print(summary)

In [ ]:
summary_df = (
    summary
    .rename_axis("Batch")
    .reset_index(name="ID_count")
)

print(summary_df)

In [ ]:
# 16 TMT channels per batch -> convert channel counts to batch counts
summary_df.Batch = (summary_df.Batch/16).astype(int)

In [ ]:
summary_df = summary_df.sort_values("Batch", ascending=False)

In [ ]:
import matplotlib.pyplot as plt

# Sort by number of batches
summary_df = summary_df.sort_values("Batch")

# Calculate percentages
summary_df["Percent"] = (
    summary_df["ID_count"] / summary_df["ID_count"].sum() * 100
)

# Visualization
plt.figure(figsize=(6, 5))

bp = summary_df.plot(
    x="Batch",
    y="Percent",
    kind="bar",
    ylim=(0, 100),
    ylabel="% proteins (total n = {})".format(summary_df["ID_count"].sum()),
    xlabel="# batches in which a protein is identified",
    color="grey",
    edgecolor="black",
    legend=False
)

# Add percentage labels
for i, v in enumerate(summary_df["Percent"]):
    plt.text(
        i,
        v + 2,
        f"{v:.2f}%",
        ha="center",
        va="bottom",
        fontname="Arial"
    )

plt.title(
    "Proteins identified across batches",
    fontname="Arial"
)

plt.xticks(fontname="Arial")
plt.yticks(fontname="Arial")

plt.tight_layout()

plt.savefig(
    "Proteins_identified_across_batches.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Convert identified batches → missed batches
summary_df["Batches_Missed"] = 6 - summary_df["Batch"]

# Sort by batches missed
summary_df = summary_df.sort_values("Batches_Missed")

# Calculate percentage
summary_df["Percent"] = (
    summary_df["ID_count"] / summary_df["ID_count"].sum() * 100
)

# Visualization
plt.figure(figsize=(3.5, 5))

bp = summary_df.plot(
    x="Batches_Missed",
    y="Percent",
    kind="bar",
    ylim=(0, 100),
    ylabel="% proteins (total n = {})".format(summary_df["ID_count"].sum()),
    xlabel="# batches in which a protein is not quantified",
    color="grey",
    edgecolor="black",
    legend=False
)

# Add percentage labels
for i, v in enumerate(summary_df["Percent"]):
    plt.text(
        i,
        v + 2,
        f"{v:.2f}%",
        ha="center",
        va="bottom",
        fontname="Arial"
    )

plt.title(
    "Proteins quantified across batches",
    fontname="Arial"
)

plt.xticks(fontname="Arial")
plt.yticks(fontname="Arial")

plt.tight_layout()

plt.savefig(
    "Proteins_quantified_across_batches2.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()